# Expressions régulières avec le module `re`

Ce notebook pratique les expressions régulières sur de petits textes, puis sur un corpus de près de 4 000 e-mails frauduleux :

1. Nettoyer un texte copié d'un PDF
2. Des numéros de téléphone au même format
3. Des liens HTML en Markdown
4. Les chaînes entre guillemets
5. Un corpus d'e-mails frauduleux

Les exercices 1 à 4 sont indépendants ; les étapes du 5 s'enchaînent. Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin.

Pour mettre au point un motif, https://regex101.com/ (saveur « Python ») explique chacune de ses parties et montre ce qu'il reconnaît. Si les bases vous manquent, le tutoriel https://regexone.com/ les reprend pas à pas.

## 1. Nettoyer un texte copié d'un PDF

Le texte `texte`, copié d'un PDF, a des blancs en trop : plusieurs espaces ou une tabulation entre deux mots, une espace avant les virgules et les points, des espaces en fin de ligne. Avec `re.sub`, construisez `propre`, le texte sans ces blancs, et affichez-le. Vous devez obtenir :

```
Le module re cherche des motifs.
Il remplace, découpe et extrait
les morceaux d'un texte.
```

Gardez les retours à la ligne : seuls les espaces et les tabulations sont en trop.

In [ ]:
import re

texte = (
  "Le  module re   cherche des motifs .\n"
  "Il\tremplace , découpe et extrait   \n"
  "les morceaux d'un texte ."
)

# Votre code ici

### Conseils supplémentaires

Trois remplacements, l'un après l'autre :

- `[ \t]+` reconnaît une suite d'espaces et de tabulations, à remplacer par une seule espace. `\s+` irait trop loin : il reconnaît aussi le saut de ligne `\n`, et recollerait les lignes
- une espace suivie d'une virgule ou d'un point : capturez la ponctuation dans un groupe, et remettez-la seule avec `\1` dans le remplacement
- les espaces juste avant une fin de ligne : `$` ne reconnaît chaque fin de ligne qu'avec le flag `re.MULTILINE`, passé à `re.sub` par son nom, `flags=`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
propre = re.sub(r"[ \t]+", " ", texte)
propre = re.sub(r"...", r"\1", propre)
propre = re.sub(r"...", "", propre, flags=...)
print(propre)
```

### Solution

In [ ]:
propre = re.sub(r"[ \t]+", " ", texte)
propre = re.sub(r" ([,.])", r"\1", propre)
propre = re.sub(r" +$", "", propre, flags=re.MULTILINE)
print(propre)

## 2. Des numéros de téléphone au même format

Un formulaire a reçu les numéros de `numeros`, écrits de six façons. Écrivez une fonction `normalise(numero)` qui renvoie le numéro au format `06 39 98 12 34` : dix chiffres groupés par deux, l'indicatif `+33` remplacé par un `0`. Affichez ensuite le numéro normalisé de chaque élément de `numeros`. Vous devez obtenir :

```
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
01 99 00 56 78
01 99 00 56 78
```

In [ ]:
numeros = [
  "06 39 98 12 34",
  "06.39.98.12.34",
  "0639981234",
  "+33 6 39 98 12 34",
  "01-99-00-56-78",
  "+33 (0)1 99 00 56 78",
]

# Votre code ici

### Conseils supplémentaires

Trois remplacements dans `normalise` :

- `\D` reconnaît ce qui n'est pas un chiffre : le supprimer ne garde que les chiffres, et `"+33 (0)1 99 00 56 78"` devient `"330199005678"`
- un `33` en début de chaîne, suivi ou non d'un `0`, devient un `0` : `^` ancre le motif au début, `?` rend le `0` facultatif
- cinq groupes de deux chiffres, `(\d{2})` cinq fois, se réécrivent `\1 \2 \3 \4 \5`

Exemple : `normalise("+33 (0)1 99 00 56 78")` renvoie `"01 99 00 56 78"`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def normalise(numero: str) -> str:
  chiffres = re.sub(r"\D", "", numero)
  chiffres = re.sub(r"...", "0", chiffres)
  return re.sub(r"...", r"\1 \2 \3 \4 \5", chiffres)


for numero in numeros:
  print(normalise(numero))
```

### Solution

In [ ]:
def normalise(numero: str) -> str:
  chiffres = re.sub(r"\D", "", numero)
  chiffres = re.sub(r"^330?", "0", chiffres)
  return re.sub(r"(\d{2})(\d{2})(\d{2})(\d{2})(\d{2})", r"\1 \2 \3 \4 \5", chiffres)


for numero in numeros:
  print(normalise(numero))

## 3. Des liens HTML en Markdown

En HTML, un lien s'écrit `<a href="adresse">texte</a>` ; en Markdown, `[texte](adresse)`. Avec `re.sub`, réécrivez en Markdown les deux liens de `html`, et affichez le résultat :

```
<p>Voir [la doc de re](https://docs.python.org/fr/3/library/re.html) et [regex101](https://regex101.com/).</p>
```

Essayez d'abord le motif `<a href="(.*)">(.*)</a>` : pourquoi ne donne-t-il pas ce résultat ? Corrigez-le.

**Documentation utile**

- [Gourmand ou non gourmand](https://docs.python.org/fr/3/howto/regex.html#greedy-versus-non-greedy)

In [ ]:
html = (
  '<p>Voir <a href="https://docs.python.org/fr/3/library/re.html">la doc de re</a>'
  ' et <a href="https://regex101.com/">regex101</a>.</p>'
)

# Votre code ici

### Conseils supplémentaires

Le remplacement réutilise les groupes dans l'autre ordre : `r"[\2](\1)"`.

`*` est gourmand : il répète le plus possible, et le premier `(.*)` va jusqu'au dernier `">` de la ligne, à travers le premier lien. Deux corrections :

- interdire le guillemet dans l'adresse et le chevron dans le texte, avec des ensembles niés : `[^"]*` et `[^<]*`
- ou rendre la répétition non gourmande, `.*?`, qui répète le moins possible

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(re.sub(r'<a href="(...)">(...)</a>', r"[\2](\1)", html))
```

### Solution

In [ ]:
# Avec des ensembles niés
print(re.sub(r'<a href="([^"]*)">([^<]*)</a>', r"[\2](\1)", html))
# Avec des répétitions non gourmandes
print(re.sub(r'<a href="(.*?)">(.*?)</a>', r"[\2](\1)", html))

## 4. Les chaînes entre guillemets

En Python, une chaîne s'écrit entre `"` ou entre `'`, et chacune peut contenir l'autre guillemet. Avec `re.finditer`, affichez le contenu de chaque chaîne de la ligne de code `ligne`, une par ligne. Vous devez obtenir :

```
Bonjour
à tous
l'été
il dit "oui"
```

Le motif `["'](.*?)["']` ne convient pas : il s'arrête à l'apostrophe de `l'été`. Capturez le guillemet ouvrant dans un groupe, et exigez le même pour fermer avec une référence arrière.

**Documentation utile**

- [Les groupes et les références arrière](https://docs.python.org/fr/3/howto/regex.html#grouping)

In [ ]:
ligne = """print("Bonjour", 'à tous', "l'été", 'il dit "oui"')"""

# Votre code ici

### Conseils supplémentaires

- dans un motif, `\1` reconnaît exactement le texte qu'a capturé le premier groupe : `(["'])` capture le guillemet ouvrant, et `\1` exige le même pour fermer
- le contenu de la chaîne est le deuxième groupe, `match.group(2)`, une répétition non gourmande `.*?` qui s'arrête au premier guillemet fermant (voir l'exercice 3)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for match in re.finditer(r"""...""", ligne):
  print(...)
```

### Solution

In [ ]:
for match in re.finditer(r"""(["'])(.*?)\1""", ligne):
  print(match.group(2))

## 5. Un corpus d'e-mails frauduleux

Le corpus CLAIR rassemble 3 976 e-mails d'arnaque « à la nigériane » (fraude 419), reçus de 1998 à 2007, à la suite les uns des autres dans un même fichier. Le fichier `debug_file.txt` n'en contient que deux, pour mettre le code au point. Source : Radev (2008), *CLAIR collection of fraud email*, ACL Data and Code Repository, ADCR2008T001, publié sur https://www.kaggle.com/datasets/rtatman/fraudulent-email-corpus sous licence CC BY-SA 4.0.

Chaque e-mail commence par une ligne `From r`, puis viennent ses en-têtes, dont :

- `From` : l'émetteur (parfois vide)
- `Reply-To` : l'adresse à laquelle répondre
- `To` : le destinataire (parfois tronqué pour l'anonymat)
- `Date` : la date d'envoi
- `Subject` : l'objet du message
- `Content-Type` : le type du contenu et son encodage

et enfin, après une ligne vide, le corps du message.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh fraud-emails

Les fichiers sont encodés en `windows-1252` : la cellule suivante les lit dans `debug_text` et `work_text`, et affiche le début du corpus.

In [ ]:
import re
from pathlib import Path

debug_text = Path("fraud-emails/debug_file.txt").read_text(encoding="windows-1252")
work_text = Path("fraud-emails/fradulent_emails.txt").read_text(encoding="windows-1252")
print(work_text[:1500])

### 5.1 Découper le corpus en e-mails

Écrivez `split_emails(text)`, qui renvoie la liste des e-mails de `text` : découpez-le avec `re.split` à chaque ligne qui commence par `From r`. Testez-la sur `debug_text`, qui doit donner 2 e-mails, puis rangez dans `emails` ceux de `work_text` : 3 976.

**Documentation utile**

- [`re.split`](https://docs.python.org/fr/3/library/re.html#re.split)

In [ ]:
def split_emails(text: str) -> list[str]: ...


# Votre code ici

#### Conseils supplémentaires

- `^` ne reconnaît chaque début de ligne qu'avec le flag `re.MULTILINE`, à passer par son nom, `flags=` (le passer en position est obsolète depuis Python 3.13)
- le fichier commence par une ligne `From r` : le premier morceau que renvoie `re.split`, ce qui la précède, est une chaîne vide, à retirer avec `[1:]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def split_emails(text: str) -> list[str]:
  return re.split(r"...", text, flags=...)[1:]


print(len(split_emails(debug_text)))
emails = split_emails(work_text)
print(len(emails))
```

#### Solution

In [ ]:
def split_emails(text: str) -> list[str]:
  return re.split(r"^From r", text, flags=re.MULTILINE)[1:]


print(len(split_emails(debug_text)))
emails = split_emails(work_text)
print(len(emails))

### 5.2 Les dix premiers émetteurs

Avec `re.findall`, affichez les dix premières lignes d'en-tête `From:` du corpus. Elles commencent par `From:`, avec un deux-points, contrairement aux lignes `From r` qui séparent les e-mails. La première est :

```
From: "MR. JAMES NGOLA." <james_ngola2002@maktoob.com>
```

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `^From:.*$` reconnaît une ligne d'en-tête entière : `.` ne reconnaît pas le saut de ligne, donc `.*` s'arrête à la fin de la ligne
- `flags=re.MULTILINE` à nouveau, pour que `^` et `$` reconnaissent chaque début et chaque fin de ligne
- `re.findall` renvoie une liste, dont `[:10]` garde les dix premiers éléments

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for line in re.findall(r"...", work_text, flags=...)[:10]:
  print(line)
```

#### Solution

In [ ]:
for line in re.findall(r"^From:.*$", work_text, flags=re.MULTILINE)[:10]:
  print(line)

### 5.3 Émetteurs et adresses

Une ligne `From:` a souvent la forme `From: "Mr. Ben Suleman" <bensul2004nng@spinfinder.com>` : un nom, parfois entre guillemets, puis une adresse entre chevrons.

1. Écrivez `process_name(name)`, qui renvoie `name` sans ses `'` ni ses `"`, ni ses espaces au début et à la fin : `process_name(' "Mr. Ben Suleman" ')` renvoie `'Mr. Ben Suleman'`.
2. Construisez le dictionnaire `senders`, qui associe à chaque nom d'émetteur, passé par `process_name`, l'ensemble de ses adresses. Gardez les lignes de la forme `nom <adresse>` dont l'adresse est plausible (des caractères sans blanc, un `@`, puis un domaine avec un point) et le nom non vide ; ignorez les autres, qui n'ont qu'une adresse ou qu'un nom.
3. Affichez le nombre de lignes `From:` gardées et ignorées, le nombre d'émetteurs, et celui des émetteurs qui ont plusieurs adresses. Avec le motif des conseils, vous devez trouver 3 190 lignes gardées, 598 ignorées, et 2 407 émetteurs, dont 162 ont plusieurs adresses.

In [ ]:
def process_name(name: str) -> str: ...


# Votre code ici

#### Conseils supplémentaires

- dans `process_name`, `re.sub` avec l'ensemble `['"]` supprime les guillemets, et la méthode `strip` des chaînes les espaces au début et à la fin
- un motif pour la ligne entière, avec un groupe pour le nom et un pour l'adresse : `^From:(.*)<(\w\S*@\S+\.\w+)>$`, toujours avec `re.MULTILINE`
- `re.finditer` donne chaque ligne reconnue, dont `match.group(1)` et `match.group(2)` sont le nom et l'adresse
- le nombre de lignes `From:` : la longueur de la liste que renvoie `re.findall(r"^From:", work_text, flags=re.MULTILINE)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def process_name(name: str) -> str:
  return re.sub(..., "", name).strip()


senders = {}
kept = 0
for match in re.finditer(r"...", work_text, flags=re.MULTILINE):
  name = process_name(match.group(1))
  if name == "":
    continue
  kept += 1
  if name not in senders:
    senders[name] = set()
  senders[name].add(...)

total = len(re.findall(r"^From:", work_text, flags=re.MULTILINE))
several = [name for name, addresses in senders.items() if len(addresses) > 1]
print(f"{kept} lignes gardées, {total - kept} ignorées")
print(f"{len(senders)} émetteurs, dont {len(several)} avec plusieurs adresses")
```

#### Solution

In [ ]:
def process_name(name: str) -> str:
  return re.sub(r"['\"]", "", name).strip()


print(repr(process_name(' "Mr. Ben Suleman" ')))

senders = {}
kept = 0
for match in re.finditer(
  r"^From:(.*)<(\w\S*@\S+\.\w+)>$", work_text, flags=re.MULTILINE
):
  name = process_name(match.group(1))
  if name == "":
    continue
  kept += 1
  if name not in senders:
    senders[name] = set()
  senders[name].add(match.group(2))

total = len(re.findall(r"^From:", work_text, flags=re.MULTILINE))
several = [name for name, addresses in senders.items() if len(addresses) > 1]
print(f"{kept} lignes gardées, {total - kept} ignorées")
print(f"{len(senders)} émetteurs, dont {len(several)} avec plusieurs adresses")

### 5.4 Les e-mails par année

Pour chaque e-mail de `emails`, cherchez l'année de son premier en-tête `Date:` avec `re.search` et un groupe nommé `year`, et comptez les e-mails de chaque année dans le dictionnaire `per_year`. Affichez les années dans l'ordre, chacune avec son nombre d'e-mails, puis le nombre d'e-mails sans année. Vous devez trouver 1 170 e-mails en 2006, l'année la plus chargée, et 566 sans année.

Le corpus va de 1998 à 2007 : d'où viennent les autres années ?

**Documentation utile**

- [Groupes nommés](https://docs.python.org/fr/3/howto/regex.html#non-capturing-and-named-groups)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- un groupe nommé s'écrit `(?P<year>...)`, et `match.group("year")` donne ce qu'il a capturé
- dans `^Date:.*?(?P<year>\d{4})`, la répétition non gourmande s'arrête au premier nombre de quatre chiffres de la ligne : `2002` dans `Date: Thu, 31 Oct 2002 05:10:00`
- `re.search` renvoie le premier résultat, ou `None` quand l'e-mail n'a pas d'en-tête `Date:` avec une année
- `per_year.get(year, 0)` vaut `0` pour une année pas encore rencontrée, et `sorted(per_year.items())` donne les paires `(année, nombre)` dans l'ordre des années

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
per_year = {}
missing = 0
for email in emails:
  match = re.search(r"...", email, flags=re.MULTILINE)
  if match is None:
    missing += 1
  else:
    year = ...
    per_year[year] = per_year.get(year, 0) + 1

for year, count in sorted(per_year.items()):
  print(year, count)
print("Sans année :", missing)
```

#### Solution

In [ ]:
per_year = {}
missing = 0
for email in emails:
  match = re.search(r"^Date:.*?(?P<year>\d{4})", email, flags=re.MULTILINE)
  if match is None:
    missing += 1
  else:
    year = match.group("year")
    per_year[year] = per_year.get(year, 0) + 1

for year, count in sorted(per_year.items()):
  print(year, count)
print("Sans année :", missing)

# Les 15 e-mails datés de 1980, 1990, 2015 ou 2082 viennent d'ordinateurs mal mis à
# l'heure (ou d'en-têtes falsifiés) : une analyse par année les écarte.